**Assignment Overview**

In this assignment, you will explore two fundamental aspects of modern NLP systems: fine-tuning large language models and understanding the attention mechanism that powers them.

**Learning Goals**

By the end of this assignment, you should be able to:

* Understand when and why to fine-tune language models
* Apply QLoRA for efficient adaptation of large models
* Analyze model outputs and limitations
* Explain and implement the attention mechanism
* Interpret attention patterns and their behavior

**Important Note**

* Do not modify or delete the task structure.
* Complete each task with:
   - Clean, well-organized code
   - Relevant visualizations
   - Clear insights and explanations
* Make sure to answer all required questions.
**Submission requirements:**
* Submit a **fully executed notebook** (all cells must run and outputs should be visible).
* There is no need to attach the training and test datasets as files, but you must present them as DataFrame tables within the notebook.


## Install Required Packages and Dataset

In [1]:
!pip install -q bitsandbytes trl
!pip install fastapi uvicorn openai
!pip install datasets
!pip install datasets openai tqdm
!pip install -U trl
!pip install -U torchao
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline,
    logging,
)
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model,PeftModel
from trl import SFTTrainer
import torch
import json
from datasets import Dataset, load_dataset
import pandas as pd


import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity(logging.CRITICAL)

## **Part 1 — QLoRA Fine-Tuning for Level-Adaptive Question Answering (75 points)**

In this part, you will fine-tune pretrained language models to answer questions at different explanation levels:

* Child — simple and intuitive explanation
* Student — clear educational explanation with moderate technical detail
* Expert — precise, technical, and domain-specific explanation

You will use a subset of the Databricks Dolly 15K dataset.

In [3]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:5000]")


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [4]:
def is_good_question(text):
    return any(q in text.lower() for q in [
        "why", "how", "what is", "explain"
    ])

filtered = [
    ex for ex in dataset
    if ex["category"] == "open_qa" and is_good_question(ex["instruction"])
]

In [5]:
len(filtered), filtered[0]

(580,
 {'instruction': 'Why can camels survive for long without water?',
  'context': '',
  'response': 'Camels use the fat in their humps to keep them filled with energy and hydration for long periods of time.',
  'category': 'open_qa'})

### **Task 1.1 — Prepare a Custom Dataset (15 points)**

Create your own instruction-tuning dataset based on a subset of databricks-dolly-15k.

For each selected question-answer pair, create versions of the answer adapted to the requested expertise level.

Example format:

    Question: What is gradient descent?
    Expertise level: child
    Answer: Gradient descent is like walking downhill step by step until you reach the lowest point.

You should prepare:

1. Training set for fine-tuning -

   Use the filtered Dolly dataset as your source of questions.

   For each question, create examples in the example format per level.

2. Test set for evaluation -
    
   Create a small test set of 20 question-answer examples.

   The test set must:

  * include examples from all three expertise levels
  * be separate from the training set
  * be manually reviewed by you for quality
  * include answers that are appropriate for the requested expertise level

  Recommendation:

  1. Save the generated dataset as a '.jsonl' file so it can be loaded and reused later.

  2. Decide on the training format according to the model architecture:
    
    - For causal language models, such as `HuggingFaceTB/SmolLM2-360M-Instruct`, use one full text field:

    ```text
    ### Question:
    ...

    ### Expertise level:
    child / student / expert

    ### Answer:
    ...```

    - For seq2seq models, such as google/flan-t5-small, separate the input and target:

    input: Question + expertise level
    target: Answer
   
  3. Before generating the full dataset, test the pipeline on a small batch of examples to verify that:

  * the LLM returns valid JSON,
  * each question receives three expertise-level answers,
  * the saved .jsonl file can be loaded correctly,
  * the format matches the training code.

In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI   

load_dotenv()

True

In [13]:
SYSTEM_PROMPT = """
You are an expert educational writer who rewrites answers at three distinct expertise levels: child, student, and expert.

You will be given a QUESTION and a reference ANSWER. Your job is to produce three rewrites of the answer — one for each level — that all answer the same question faithfully, but in voices appropriate to the audience.

LEVEL DEFINITIONS

- child: A curious 7-year-old. Use everyday words and simple sentences. Use a concrete analogy (something a child has seen — toys, food, animals, weather). No jargon, no symbols, no numbers unless tiny. 2–3 short sentences.

- student: An undergraduate encountering the topic for the first time. Define any technical term you use in the same sentence. Give the core mechanism plus one small example. Field-standard vocabulary is allowed but always glossed. 3–5 sentences.

- expert: A practitioner in the field. Use precise technical terminology without explaining it. Be dense, exact, and compact. Skip introductions and analogies. 2–4 sentences.

RULES

1. All three answers must remain faithful to the reference answer. Do not invent new facts or contradict it.
2. The three answers must be clearly different in voice and vocabulary — not just length.
3. The child version must contain no technical jargon at all.
4. The expert version must contain no hand-holding ("simply put", "in other words", analogies).
5. Output ONLY valid JSON with exactly three string keys: "child", "student", "expert". No markdown fences, no commentary, no preamble.

SELF-CHECK before producing output: If you read the three answers without their labels, would a stranger correctly guess which is which? If not, rewrite to make the voice differences sharper.

OUTPUT FORMAT

{"child": "...", "student": "...", "expert": "..."}
"""


In [14]:
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv("NEBIUS_API_KEY"),
)

In [ ]:
from pydantic import BaseModel


class LevelAnswers(BaseModel):
    child: str
    student: str
    expert: str
    
def make_user_message(question: str, answer: str) -> str:
    return f"""QUESTION: {question}\n\nANSWER: {answer}"""
    
    
def generate_level_answers(question: str, answer: str) -> dict:
    response = client.chat.completions.parse(
        model="moonshotai/Kimi-K2.5",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": make_user_message(question, answer)
            }
        ],
        temperature=0.7,
        response_format=LevelAnswers
    )
    return response.choices[0].message.parsed


Example 1
Question: Why can camels survive for long without water?
Reference Answer: Camels use the fat in their humps to keep them filled with energy and hydration for long periods of time.
Parsed Answers: child='Camels have a big bump on their back filled with fat, like a backpack full of snacks. When they get hungry or thirsty, their body turns that fat into energy and water. This lets them walk through the hot desert for days without drinking.' student='Camels store adipose tissue (body fat) concentrated in their dorsal hump, which serves as a metabolic reservoir. When the animal cannot find food or water, enzymes break down this fat through beta-oxidation, a process that releases both calories for energy and metabolic water—about one gram of water for every gram of fat burned. This physiological adaptation allows camels to maintain internal stability during long periods of drought.' expert='Camels utilize concentrated adipose deposits in their dorsal hump as a metabolic energy res

In [27]:
import random
random.seed(42)
shuffled = filtered.copy()
random.shuffle(shuffled)
test_questions  = shuffled[:20]
train_questions = shuffled[20:]

In [28]:

import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from tqdm import tqdm

file_lock = Lock()

def process_one(ex, fout):
    try:
        levels = generate_level_answers(ex["instruction"], ex["response"])
    except Exception as e:
        return ("error", ex["instruction"][:60], str(e))

    rows = [
        {"question": ex["instruction"], "level": lvl,
         "answer": getattr(levels, lvl)}
        for lvl in ("child", "student", "expert")
    ]

    with file_lock:
        for row in rows:
            fout.write(json.dumps(row) + "\n")
        fout.flush()

    return ("ok", ex["instruction"][:60], None)


with open("train_master.jsonl", "w") as fout, ThreadPoolExecutor(max_workers=12) as pool:

    futures = [pool.submit(process_one, ex, fout) for ex in train_questions]

    for fut in tqdm(as_completed(futures), total=len(futures)):
        status, q, err = fut.result()
        if status == "error":
            print(f"error processing:{q!r} — {err}")

 62%|██████▎   | 350/560 [11:36<05:58,  1.71s/it]

error processing:'What is a prime number?' — 1 validation error for LevelAnswers
  Invalid JSON: invalid escape at line 1 column 773 [type=json_invalid, input_value=' {"child": "A prime numb...uniquely into primes."}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid


 93%|█████████▎| 521/560 [15:35<00:43,  1.12s/it]

error processing:'What is the difference between natural and real numbers?' — 1 validation error for LevelAnswers
  Invalid JSON: invalid escape at line 1 column 369 [type=json_invalid, input_value=' {"child": "Natural numb...l non-integer values."}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid


100%|██████████| 560/560 [16:18<00:00,  1.75s/it]


In [29]:
with open("test_master.jsonl", "w") as fout, ThreadPoolExecutor(max_workers=6) as pool:
    futures = [pool.submit(process_one, ex, fout) for ex in test_questions]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        status, q, err = fut.result()
        if status == "error":
            print(f"error processing: {q!r} — {err}")
            

with open("train_master.jsonl") as fin, open("train_causal.jsonl", "w") as fout:
    for line in fin:
        r = json.loads(line)
        text = f"### Question:\n{r['question']}\n\n### Expertise level:\n{r['level']}\n\n### Answer:\n{r['answer']}"
        fout.write(json.dumps({"text": text}) + "\n")
        
with open("train_master.jsonl") as fin, open("train_seq2seq.jsonl", "w") as fout:
    for line in fin:
        r = json.loads(line)
        fout.write(json.dumps({
            "input":  f"Question: {r['question']} Expertise level: {r['level']}",
            "target": r["answer"],
        }) + "\n")
        


100%|██████████| 20/20 [00:40<00:00,  2.01s/it]


In [30]:
import pandas as pd
df_train = pd.read_json("train_master.jsonl", lines=True)
print(f"Training set: {len(df_train)} rows")
df_train.head(9)

Training set: 1674 rows


,question,level,answer
0,What's the most popular tv show of all time in...,child,The most popular TV show ever in America is ca...
1,What's the most popular tv show of all time in...,student,Jeopardy! is widely considered the most popula...
2,What's the most popular tv show of all time in...,expert,Jeopardy! maintains the highest cumulative Nie...
3,What is net worth?,child,Think about your piggy bank. If you count all ...
4,What is net worth?,student,Net worth quantifies an entity's equity by sub...
5,What is net worth?,expert,Net worth constitutes equity—the residual clai...
6,What is the team that Michael Jordan played?,child,Michael Jordan played for the Chicago Bulls. T...
7,What is the team that Michael Jordan played?,student,"Michael Jordan played for the Chicago Bulls, a..."
8,What is the team that Michael Jordan played?,expert,"Michael Jordan played for the Chicago Bulls, s..."


### **Task 1.2 — Fine-Tune Models with QLoRA (10 points)**

Fine-tune the models using QLoRA, meaning:

1. Load the base model in 4-bit quantization
2. Add LoRA adapters
3. Train only the adapter parameters

You should repeat the fine-tuning process for both models:

1. HuggingFaceTB/SmolLM2-360M-Instruct
2. google/flan-t5-small

Pay attention each model requires the dataset to be formated diffrently.

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

In [7]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 819,200 || all params: 362,640,320 || trainable%: 0.2259


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
train_dataset = Dataset.from_json("/content/drive/MyDrive/nebius-hw3/train_causal.jsonl")
print(train_dataset)
print("\n--- example row ---")
print(train_dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1674
})

--- example row ---
### Question:
What's the most popular tv show of all time in the USA?

### Expertise level:
child

### Answer:
The most popular TV show ever in America is called Jeopardy! It is a quiz show where people try to answer tricky questions to win money. It has been on television almost every day for longer than most people can remember!


In [11]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/nebius-hw3/smollm2-qlora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    bf16=False,
    fp16=False,
    optim="adamw_torch",
    logging_steps=25,
    save_strategy="epoch",
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

adapter_dir = "/content/drive/MyDrive/nebius-hw3/smollm2-qlora-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"adapter saved to {adapter_dir}")

Adding EOS to train dataset:   0%|          | 0/1674 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1674 [00:00<?, ? examples/s]

{'loss': '2.919', 'grad_norm': '0.4395', 'learning_rate': '0.0001833', 'entropy': '2.329', 'num_tokens': '4.312e+04', 'mean_token_accuracy': '0.4538', 'epoch': '0.2387'}
{'loss': '2.592', 'grad_norm': '0.2754', 'learning_rate': '0.0001586', 'entropy': '2.248', 'num_tokens': '8.576e+04', 'mean_token_accuracy': '0.4985', 'epoch': '0.4773'}
{'loss': '2.42', 'grad_norm': '0.2637', 'learning_rate': '0.000134', 'entropy': '2.215', 'num_tokens': '1.287e+05', 'mean_token_accuracy': '0.5239', 'epoch': '0.716'}
{'loss': '2.36', 'grad_norm': '0.2793', 'learning_rate': '0.0001094', 'entropy': '2.264', 'num_tokens': '1.714e+05', 'mean_token_accuracy': '0.53', 'epoch': '0.9547'}
{'loss': '2.287', 'grad_norm': '0.3145', 'learning_rate': '8.473e-05', 'entropy': '2.232', 'num_tokens': '2.135e+05', 'mean_token_accuracy': '0.5416', 'epoch': '1.191'}
{'loss': '2.27', 'grad_norm': '0.3086', 'learning_rate': '6.01e-05', 'entropy': '2.234', 'num_tokens': '2.561e+05', 'mean_token_accuracy': '0.5376', 'epoch':

### Repeat: flan-t5-small (seq2seq) — same QLoRA recipe, four pieces change

In [14]:
bnb_config_t5 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

t5_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small",
    quantization_config=bnb_config_t5,
    device_map="auto",
)
t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = prepare_model_for_kbit_training(t5_model)

flan_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=["q", "v"],
)

t5_model = get_peft_model(t5_model, flan_lora_config)
t5_model.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


In [15]:
raw_t5_dataset = Dataset.from_json("/content/drive/MyDrive/nebius-hw3/train_seq2seq.jsonl")

def tokenize_seq2seq(batch):
    model_inputs = t5_tokenizer(batch["input"], max_length=256, truncation=True)
    labels = t5_tokenizer(text_target=batch["target"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_t5 = raw_t5_dataset.map(
    tokenize_seq2seq,
    batched=True,
    remove_columns=["input", "target"],
)

t5_split = tokenized_t5.train_test_split(test_size=0.1, seed=42)
t5_train_dataset = t5_split["train"]
t5_eval_dataset = t5_split["test"]

print(f"train: {len(t5_train_dataset)} | eval: {len(t5_eval_dataset)}")

train: 1506 | eval: 168


In [16]:
flan_training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/nebius-hw3/flan-t5-small-qlora",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    num_train_epochs=7,

    logging_strategy="steps",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    predict_with_generate=True,
    generation_max_length=128,

    bf16=False,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
)

t5_data_collator = DataCollatorForSeq2Seq(
    tokenizer=t5_tokenizer,
    model=t5_model,
    label_pad_token_id=-100,
    padding="longest",
)

t5_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=flan_training_args,
    train_dataset=t5_train_dataset,
    eval_dataset=t5_eval_dataset,
    data_collator=t5_data_collator,
    processing_class=t5_tokenizer,
)

t5_trainer.train()

t5_adapter_dir = "/content/drive/MyDrive/nebius-hw3/flan-t5-small-qlora-adapter"
t5_model.save_pretrained(t5_adapter_dir)
t5_tokenizer.save_pretrained(t5_adapter_dir)
print(f"adapter saved to {t5_adapter_dir}")

{'loss': '9.806', 'grad_norm': '0.02153', 'learning_rate': '9.94e-05', 'epoch': '0.05263'}
{'loss': '9.8', 'grad_norm': '0.0218', 'learning_rate': '9.865e-05', 'epoch': '0.1053'}
{'loss': '9.799', 'grad_norm': '0.02487', 'learning_rate': '9.789e-05', 'epoch': '0.1579'}
{'loss': '9.799', 'grad_norm': '0.02788', 'learning_rate': '9.714e-05', 'epoch': '0.2105'}
{'loss': '9.794', 'grad_norm': '0.03197', 'learning_rate': '9.639e-05', 'epoch': '0.2632'}
{'eval_loss': '9.753', 'eval_runtime': '1.176', 'eval_samples_per_second': '142.8', 'eval_steps_per_second': '9.35', 'epoch': '0.2632'}
{'loss': '9.798', 'grad_norm': '0.05409', 'learning_rate': '9.564e-05', 'epoch': '0.3158'}
{'loss': '9.79', 'grad_norm': '0.04195', 'learning_rate': '9.489e-05', 'epoch': '0.3684'}
{'loss': '9.781', 'grad_norm': '0.04115', 'learning_rate': '9.414e-05', 'epoch': '0.4211'}
{'loss': '9.777', 'grad_norm': '0.043', 'learning_rate': '9.338e-05', 'epoch': '0.4737'}
{'loss': '9.771', 'grad_norm': '0.04608', 'learning

### **Task 1.3 — Evaluation (20 points)**

Evaluate both fine-tuned models on your 20-example test set.

For each model, compare:

* outputs before fine-tuning
* outputs after fine-tuning
* whether the answer matches the requested expertise level
* whether the answer is clear and relevant
* whether the answer stays faithful to the question

Evaluate using this table:

|Question|Level|Base Output|Fine-tuned Output|Expected Answer|Better after FT?|Notes|
|-------|-------|-------|-------|-------|-------|-------|


In [17]:
from tqdm.notebook import tqdm

eval_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

smol_base = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    quantization_config=eval_bnb_config,
    device_map="auto",
)
smol_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
smol_tokenizer.pad_token = smol_tokenizer.eos_token
smol_ft = PeftModel.from_pretrained(
    smol_base,
    "/content/drive/MyDrive/nebius-hw3/smollm2-qlora-adapter",
)
smol_ft.eval()

t5_base = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small",
    quantization_config=eval_bnb_config,
    device_map="auto",
)
t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_ft = PeftModel.from_pretrained(
    t5_base,
    "/content/drive/MyDrive/nebius-hw3/flan-t5-small-qlora-adapter",
)
t5_ft.eval()


def generate_smollm(model, question, level, max_new_tokens=120):
    prompt = f"### Question:\n{question}\n\n### Expertise level:\n{level}\n\n### Answer:\n"
    inputs = smol_tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=smol_tokenizer.eos_token_id,
        )
    return smol_tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


def generate_t5(model, question, level, max_new_tokens=120):
    prompt = f"Question: {question} Expertise level: {level}"
    inputs = t5_tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return t5_tok.decode(out[0], skip_special_tokens=True).strip()


test_set = []
with open("/content/drive/MyDrive/nebius-hw3/test_master.jsonl") as f:
    for line in f:
        test_set.append(json.loads(line))

print(f"test set: {len(test_set)} examples ({len(test_set)//3} questions x 3 levels)")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

test set: 60 examples (20 questions x 3 levels)


In [18]:
smol_rows = []
for ex in tqdm(test_set, desc="SmolLM2"):
    q, level, expected = ex["question"], ex["level"], ex["answer"]
    ft_out = generate_smollm(smol_ft, q, level)
    with smol_ft.disable_adapter():
        base_out = generate_smollm(smol_ft, q, level)
    smol_rows.append({
        "Question": q,
        "Level": level,
        "Base Output": base_out,
        "Fine-tuned Output": ft_out,
        "Expected Answer": expected,
        "Better after FT?": "",
        "Notes": "",
    })

df_smol_eval = pd.DataFrame(smol_rows)
df_smol_eval.to_csv("/content/drive/MyDrive/nebius-hw3/smollm2_eval.csv", index=False)
print(f"saved {len(df_smol_eval)} rows to smollm2_eval.csv")
df_smol_eval.head(6)

SmolLM2:   0%|          | 0/60 [00:00<?, ?it/s]

saved 60 rows to smollm2_eval.csv


,Question,Level,Base Output,Fine-tuned Output,Expected Answer,Better after FT?,Notes
0,What is the best sport?,child,The best sport is basketball.\n\nQuestion: Wha...,"The best sport is basketball. It's fun, easy t...","There is no single best sport, just like there...",,
1,What is the best sport?,student,The best sport is basketball.\n\nQuestion: Wha...,The best sport is basketball. It is played wit...,There is no objectively 'best' sport because t...,,
2,What is the best sport?,expert,The best sport is basketball.\n\nQuestion: Wha...,The best sport is the one that is played with ...,Sport preference is a subjective utility funct...,,
3,Why do people eat food?,child,People eat food because it provides them with ...,People eat food to get energy for their bodies...,Your body works like a toy car that needs batt...,,
4,Why do people eat food?,student,People eat food because it provides them with ...,"People eat food to obtain energy, which is the...",Food supplies chemical energy through cellular...,,
5,Why do people eat food?,expert,People eat food because it provides them with ...,The primary purpose of consuming food is to pr...,Dietary substrates undergo catabolism via oxid...,,


In [19]:
t5_rows = []
for ex in tqdm(test_set, desc="flan-t5-small"):
    q, level, expected = ex["question"], ex["level"], ex["answer"]
    ft_out = generate_t5(t5_ft, q, level)
    with t5_ft.disable_adapter():
        base_out = generate_t5(t5_ft, q, level)
    t5_rows.append({
        "Question": q,
        "Level": level,
        "Base Output": base_out,
        "Fine-tuned Output": ft_out,
        "Expected Answer": expected,
        "Better after FT?": "",
        "Notes": "",
    })

df_t5_eval = pd.DataFrame(t5_rows)
df_t5_eval.to_csv("/content/drive/MyDrive/nebius-hw3/flan_t5_eval.csv", index=False)
print(f"saved {len(df_t5_eval)} rows to flan_t5_eval.csv")
df_t5_eval.head(6)

flan-t5-small:   0%|          | 0/60 [00:00<?, ?it/s]

saved 60 rows to flan_t5_eval.csv


,Question,Level,Base Output,Fine-tuned Output,Expected Answer,Better after FT?,Notes
0,What is the best sport?,child,sport,sport is a form of a form of a form of a form ...,"There is no single best sport, just like there...",,
1,What is the best sport?,student,acrobatics,sport is a form of a form of a form of a form ...,There is no objectively 'best' sport because t...,,
2,What is the best sport?,expert,acrobatics,sport is a form of a form of a form of a form ...,Sport preference is a subjective utility funct...,,
3,Why do people eat food?,child,food is a food,food is a food that is a food that is a food t...,Your body works like a toy car that needs batt...,,
4,Why do people eat food?,student,eats food,food is a food that is a food that is a food t...,Food supplies chemical energy through cellular...,,
5,Why do people eat food?,expert,level,food is a food that is a food that is a food t...,Dietary substrates undergo catabolism via oxid...,,


### **Task 1.4 — Model Comparison and Discussion (15 points)**

Compare the performance of the two models:

* Which model adapted better to the expertise levels?
* Which model produced clearer answers?
* Which model followed the requested format better?
* Did one model hallucinate more than the other?

Explain any differences you observe.

In your discussion, consider that:

* SmolLM2-360M-Instruct is a decoder-only instruction model
* flan-t5-small is an encoder-decoder instruction model
* Different architectures may behave differently on instruction-following and text generation tasks

The two models behaved very differently after QLoRA fine-tuning, and the differences are explained almost entirely by architecture and scale.

**Level adaptation.** SmolLM2 (decoder-only, 360M parameters) clearly learned to vary its register with the requested expertise level. The base model produced an identical answer to a given question regardless of whether `child`, `student`, or `expert` was specified — for "Why do people eat food?" all three levels returned the same response. After two epochs of QLoRA training, the same prompt produces three distinct outputs: *"get energy for their bodies"* (child) → *"obtain energy, which is the..."* (student) → *"The primary purpose of consuming food is to provide..."* (expert). The model is picking up the expertise tag and modulating vocabulary accordingly. flan-t5-small (encoder-decoder, 77M parameters) showed no such adaptation. Its base outputs were already degenerate (`"sport"`, `"acrobatics"`, `"food is a food"`), and after fine-tuning it collapsed into pure repetition loops (`"sport is a form of a form of a form..."`). It did not learn the task.

**Clarity and format compliance.** Fine-tuned SmolLM2 outputs are coherent, full-sentence answers in the expected register, with no obvious format violations. Fine-tuned flan-t5-small outputs are mostly unparsable strings of repeated phrases. The base T5 outputs were already short and incoherent — fine-tuning made them worse, not better.

**Hallucination.** Both base models hallucinated content (SmolLM2 confidently asserted *"the best sport is basketball"*; T5 emitted random fragments like *"acrobatics"*). After fine-tuning, SmolLM2 still hallucinated some facts but maintained coherent answer structure and the requested register; T5 stopped producing meaningful content entirely. So the kind of hallucination is different — SmolLM2 hallucinates *content within a correct format*, while fine-tuned T5 hallucinates *the entire output space*.

**Why the gap.** Three compounding factors explain the failure on T5. *(1) Scale*: flan-t5-small has ~5x fewer parameters than SmolLM2, with proportionally less representational capacity to absorb a new conditioning behavior on top of pretraining. *(2) Architecture*: encoder-decoder models route information twice (encoder → cross-attention → decoder), which amplifies any noise introduced by quantization. The 4-bit NF4 weights distort T5's RMSNorm scaling and the cross-attention projections, and a small T5 has no redundancy to recover from that. *(3) Training signal*: T5's final train_loss of 9.33 sits barely below random prediction on a 32K-vocab model (ln 32128 ≈ 10.4), meaning the gradient signal was effectively noise — the LoRA adapter drifted into the repetition basin rather than learning the conditioning task. Meanwhile SmolLM2's loss dropped from 2.92 to 2.27 over 210 steps, a healthy 22% reduction that translated into visible behavioral change. The lesson is that QLoRA is not architecture-agnostic: it fits instruction-tuned decoder-only models in the few-hundred-million-parameter range cleanly, and a small encoder-decoder is the wrong target for the same recipe.

### **Task 1.5 - Conceptual Questions (15 points)**

Answer the following questions:

1. What changed after fine-tuning?
   
   Discuss whether the model became better at adapting its answer to the requested expertise level.
2. Why is QLoRA more memory efficient?
   
   Explain the role of 4-bit quantization and LoRA adapters.
3. What happens if you increase the LoRA rank?
   
   Discuss the tradeoff between model capacity, memory usage, and overfitting risk.
4. Why use LoRA / QLoRA instead of prompt engineering?

      Discuss:

      * In what cases prompt engineering is sufficient
      * When fine-tuning becomes necessary
      * What advantages QLoRA provides over prompting
      * What are the trade-offs (cost, flexibility, control)

**1. What changed after fine-tuning?**

For SmolLM2, fine-tuning produced a clear, measurable change in behavior. Before fine-tuning, the base model gave the same answer regardless of the requested level — the level tag in the prompt was being ignored entirely. After two epochs of QLoRA training on 1674 level-labeled examples, the model produces three distinct outputs per question: the child version uses simple verbs and short clauses (*"get energy for their bodies"*), the student version introduces brief technical framing (*"obtain energy, which is the..."*), and the expert version adopts a formal register (*"The primary purpose of consuming food is..."*). The training-loss curve supports this: loss fell from 2.92 at step 25 to 2.27 by step 150 and then plateaued, indicating the model had absorbed the conditioning behavior available at this LoRA capacity. For flan-t5-small the answer is the opposite — fine-tuning made things worse, because the train_loss never dropped below 9.3 on a 32K-vocab model where random prediction sits at ~10.4. So fine-tuning *can* teach a model new conditioning behavior, but only when the architecture, scale, and quantization scheme are compatible.

**2. Why is QLoRA more memory efficient?**

QLoRA stacks two memory savings that compound multiplicatively. *4-bit quantization* stores each base-model weight in one of 16 codepoints, chosen via NF4 to match the empirical Gaussian-like distribution of neural-network weights. For SmolLM2-360M the full-precision weights would occupy ~1.4 GB; in 4-bit with double quantization the same weights fit in roughly 250 MB — a 5–6x reduction. *LoRA* then freezes the entire base model and trains only a pair of low-rank matrices attached to selected attention projections (A is `r × in_features`, B is `out_features × r`). In our SmolLM2 run this gave **819,200 trainable parameters out of 362,640,320 total — 0.226% of the model**. The optimizer state, normally ~2x the size of the trainable parameters in fp32, now scales only with the LoRA path and not with the frozen base. The combined effect: a 360M model that would not fit on a free Colab T4 in full precision trains comfortably in 4-bit + LoRA, and only the 3 MB adapter needs to be saved or shipped.

**3. What happens if you increase the LoRA rank?**

Raising the rank `r` increases the capacity of the adapter linearly: at `r=8` each attention projection adds `8 × (d_in + d_out)` trainable parameters; at `r=16` it adds twice that; at `r=64` eight times. With more capacity the adapter can represent more complex deltas to the base model, which helps when the target task is substantially different from what the base was pretrained on. The trade-offs are three. *Memory* grows linearly with `r` — both the parameter count and the matching optimizer state. *Overfitting risk* also grows: on a small dataset like ours (1674 examples), an `r=64` adapter has enough degrees of freedom to memorize answers rather than generalize the level-conditioning pattern. *Diminishing returns*: in our SmolLM2 run the loss plateaued at ~2.27 from step 150 onward even with `r=8`, suggesting the bottleneck was not adapter capacity but the dataset size and the inherent difficulty of three-way register conditioning. Bumping `r` higher would more likely have produced memorization than improvement here. The right move when raising `r` is usually to keep `lora_alpha = 2*r` so the effective adapter scaling stays constant, and to add regularization (more dropout, fewer epochs) to manage the new overfitting headroom.

**4. Why use LoRA / QLoRA instead of prompt engineering?**

Prompt engineering is sufficient when the task can be solved by *showing* the model what to do — a few in-context examples, a system message, or carefully worded instructions that activate behavior the base model already knows. It is the right choice when behavior needs to vary per call, when no labeled training data exists, or when the deployment context allows full prompt control. Fine-tuning becomes necessary when **(a)** the desired behavior is consistent across many calls and re-spelling it in every prompt wastes tokens, **(b)** prompt length runs into context-window or latency limits, or **(c)** the task requires the model to internalize a pattern that few-shot examples cannot reliably elicit — producing answers in a specific register, routing to a custom output schema, learning a domain vocabulary. QLoRA's specific advantages are *cost* (a 360M model trains in ~7 minutes on a free GPU, with a 3 MB resulting adapter), *modularity* (different adapters can be swapped on the same base model at inference time, letting one base serve many specialized behaviors), and *predictability* (the model emits the trained register without the prompt babysitting that few-shot needs). The trade-offs are real: training requires curated labeled data, the adapter is locked to its training distribution, and quality depends heavily on choices like rank, learning rate, and target modules — as our flan-t5-small failure makes vivid. Prompt engineering and QLoRA are not substitutes but complements: prompt-engineer first to confirm the task is even tractable on the base model, then fine-tune to bake in the behaviors that prompts kept producing inconsistently.

## **Part 2 — Understanding Attention (25 points)**

Goal:

Build intuition for how attention works.

Task:

You will implement a simple attention mechanism from scratch (PyTorch) and visualize its behavior.

### **Task 2.1 Implement Scaled Dot-Product Attention (5 points)**

Given:

* Query (Q)
* Key (K)
* Value (V)

Compute:

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Implements Scaled Dot-Product Attention.

    Args:
        Q: (batch, heads, seq_len_q, d_k)
        K: (batch, heads, seq_len_k, d_k)
        V: (batch, heads, seq_len_v, d_v)
        mask: (batch, heads, seq_len_q, seq_len_k) or None

    Returns:
        output: (batch, heads, seq_len_q, d_v)
        attention_weights: (batch, heads, seq_len_q, seq_len_k)
    """


    #  Get dimension for scaling
    d_k = Q.size(-1)


    # Compute attention scores

    scores = # TODO: compute dot-product between Q and K^T


    # Scale the scores

    # TODO: divide scores by sqrt(d_k)


    #  Apply mask (if given)

    if mask is not None:
        # TODO: mask out invalid positions (set to -inf)
        pass


    # Softmax to get attention

    attention_weights = # TODO: apply softmax over last dimension


    # Compute weighted sum

    output = #TODO: multiply attention_weights with V

    return output, attention_weights

### **Task 2.2 Visualize Attention (5 points)**


Plot attention weights as a heatmap for the given sentence


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import math



sentence = "the cat sat on the mat"


tokens = # TODO: split sentence into tokens

seq_len = # TODO: compute sequence length



# Create dummy embeddings

d_model = 16


embeddings = # TODO: initialize random embeddings of shape (seq_len, d_model)

# Treat embeddings as Q, K, V
Q = embeddings
K = embeddings
V = embeddings



# Compute Scaled Dot-Product Attention

attention_weights = # TODO: compute attention weights (use function from above)



# Plot attention heatmap

plt.figure(figsize=(8, 6))

# TODO: plot heatmap using seaborn
# - use attention_weights
# - set xticklabels and yticklabels to tokens
# - choose a colormap (e.g., "viridis")

plt.title("Attention Weights Heatmap")

# TODO: label axes
# plt.xlabel(...)
# plt.ylabel(...)

# TODO: rotate ticks if needed

plt.show()



### **Task 2.3 Experiments (15 points)**

**Experiment 1 — Change One Word:**

1. Use a simple sentence, for example:
    the cat sat on the mat
2. Compute and plot the attention weights as a heatmap.
3. Change one word in the sentence, for example:
    the dog sat on the mat
4. Recompute and plot the attention heatmap.
5. Compare the two heatmaps and explain whether the attention pattern changed.

**Experiment 2 — Compare Different Attention Heads:**

Repeat the attention visualization using at least two different attention heads.

For each head, create separate projection matrices:

$$W_Q, W_K, W_V$$

Use them to compute:
$$Q=XW_Q, K=XW_K, V=XW_V$$

Then compute and plot the attention weights for each head.

**Questions**

Answer briefly:

1. Did changing one word affect the attention weights? Why or why not?
2. Do different attention heads focus on different tokens?
3. Why might multi-head attention be useful in Transformer models?